# Decision Tree Baseline — SSU Problem Solving 2026
목표: 3개 feature tier × DecisionTree CV 비교 → 최적 tier 하이퍼파라미터 튜닝 → 해석/산출물 저장

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score, accuracy_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
print('Libraries loaded.')

## 1. 데이터 로드

In [ ]:
# 로컬 파일 우선, 없으면 kagglehub 다운로드
LOCAL_TRAIN = 'train.csv'
LOCAL_TEST  = 'test.csv'

if os.path.exists(LOCAL_TRAIN):
    train = pd.read_csv(LOCAL_TRAIN)
    test  = pd.read_csv(LOCAL_TEST) if os.path.exists(LOCAL_TEST) else None
    print(f'Local files loaded. train: {train.shape}')
    if test is not None:
        print(f'test: {test.shape}')
else:
    import kagglehub
    path = kagglehub.competition_download('ssu-problem-solving-2026')
    print('Path to competition files:', path)
    train = pd.read_csv(os.path.join(path, 'train.csv'))
    test_path = os.path.join(path, 'test.csv')
    test = pd.read_csv(test_path) if os.path.exists(test_path) else None
    print(f'train: {train.shape}')
    if test is not None:
        print(f'test: {test.shape}')

## 2. Sanity Check

In [ ]:
print('=== train shape:', train.shape)
print('\n=== 결측치:')
print(train.isnull().sum()[train.isnull().sum() > 0] if train.isnull().sum().sum() > 0 else '  결측치 없음')

print('\n=== target 분포:')
vc = train['target'].value_counts()
print(vc)
print(f'  base rate (class 0) = {vc[0]/len(train):.4f}  → accuracy 단독 평가 금지')

print('\n=== dtype 분류:')
num_cols = [c for c in train.columns if train[c].dtype != object and c not in ['id', 'target']]
cat_cols = [c for c in train.columns if train[c].dtype == object]
print(f'  수치형 {len(num_cols)}개: {num_cols}')
print(f'  범주형 {len(cat_cols)}개: {cat_cols}')

print('\n=== 중복행:', train.duplicated().sum())

## 3. Feature Tier 정의

In [ ]:
DROP_ID_LIKE = ['id', 'feat_34', 'feat_08', 'feat_14']   # ID성/자기참조 feature

CORE6 = ['feat_01', 'feat_02', 'feat_05', 'feat_07', 'feat_31', 'feat_36']
CORE7 = CORE6 + ['feat_03']

# 전체 34개 (DROP_ID_LIKE 제거 후)
ALL_FEATS = [c for c in train.columns if c not in DROP_ID_LIKE + ['target']]

TIERS = {
    'ALL34': ALL_FEATS,
    'CORE6': CORE6,
    'CORE7': CORE7,
}

# 각 tier의 범주형 컬럼
def get_cat_cols(feature_list):
    return [c for c in feature_list if train[c].dtype == object]

print('Tier별 feature 수:')
for name, feats in TIERS.items():
    print(f'  {name}: {len(feats)}개  (범주형: {get_cat_cols(feats)})')

# tier 컬럼 존재 확인
for name, feats in TIERS.items():
    missing = [f for f in feats if f not in train.columns]
    if missing:
        print(f'  WARNING {name}에 없는 컬럼: {missing}')
    else:
        print(f'  {name} 컬럼 모두 존재 ✓')

## 4. 전처리 파이프라인 & 평가 함수

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def build_pipeline(feature_cols, model):
    """OrdinalEncoder(범주형)만 적용하는 파이프라인. 수치형은 pass-through."""
    cat = get_cat_cols(feature_cols)
    num = [c for c in feature_cols if c not in cat]

    if cat:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        ct = ColumnTransformer(
            [('ord', enc, cat)],
            remainder='passthrough',
            verbose_feature_names_out=False
        )
        pipe = Pipeline([('prep', ct), ('clf', model)])
    else:
        pipe = Pipeline([('clf', model)])
    return pipe


def evaluate(model, feature_cols, cv=CV):
    """5-fold CV로 AUC/balanced_acc/F1/acc 반환 (평균±std)."""
    X = train[feature_cols]
    y = train['target']

    pipe = build_pipeline(feature_cols, model)

    scoring = {
        'roc_auc':           'roc_auc',
        'balanced_accuracy': 'balanced_accuracy',
        'f1_macro':          'f1_macro',
        'accuracy':          'accuracy',
    }
    cv_res = cross_validate(pipe, X, y, cv=cv, scoring=scoring,
                            return_train_score=True, n_jobs=-1)

    result = {}
    for metric in scoring:
        cv_scores   = cv_res[f'test_{metric}']
        train_scores = cv_res[f'train_{metric}']
        result[metric] = {
            'cv_mean': cv_scores.mean(),
            'cv_std':  cv_scores.std(),
            'train_mean': train_scores.mean(),
            'gap': train_scores.mean() - cv_scores.mean(),
        }
    return result


def print_results(name, result):
    print(f'\n── {name} ──')
    print(f"  {'Metric':<20} {'CV mean':>8} {'±std':>7} {'Train':>8} {'Gap':>7}")
    print(f"  {'-'*54}")
    for m, v in result.items():
        print(f"  {m:<20} {v['cv_mean']:>8.4f} {v['cv_std']:>7.4f} {v['train_mean']:>8.4f} {v['gap']:>7.4f}")

print('평가 함수 정의 완료.')

## 5. Decision Tree Baseline — 3 Tier 비교

In [ ]:
# 기본 DT (default: depth 무제한 → 과적합 확인용)
baseline_results = {}

for tier_name, feats in TIERS.items():
    model = DecisionTreeClassifier(random_state=RANDOM_STATE)
    res = evaluate(model, feats)
    baseline_results[tier_name] = res
    print_results(f'{tier_name} (default DT, depth=None)', res)

print('\n[관찰] depth=None이면 Train AUC≈1.0, CV AUC는 크게 낮음 → 과적합 심각')

In [ ]:
# Tier별 CV AUC 요약 시각화
tier_names = list(TIERS.keys())
cv_aucs  = [baseline_results[t]['roc_auc']['cv_mean']  for t in tier_names]
cv_stds  = [baseline_results[t]['roc_auc']['cv_std']   for t in tier_names]
tr_aucs  = [baseline_results[t]['roc_auc']['train_mean'] for t in tier_names]

x = np.arange(len(tier_names))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, tr_aucs, 0.35, label='Train AUC', color='steelblue', alpha=0.7)
ax.bar(x + 0.2, cv_aucs, 0.35, yerr=cv_stds, label='CV AUC (±std)', color='coral', alpha=0.9, capsize=5)
ax.set_xticks(x); ax.set_xticklabels(tier_names)
ax.set_ylabel('ROC-AUC'); ax.set_title('Baseline DT (depth=None): Train vs CV AUC by Tier')
ax.legend(); ax.set_ylim(0.5, 1.05)
plt.tight_layout()
plt.savefig('baseline_tier_comparison.png', bbox_inches='tight')
plt.show()
print('Saved: baseline_tier_comparison.png')

## 6. 하이퍼파라미터 튜닝 (최적 Tier)

In [ ]:
# CV AUC 기준으로 최적 tier 자동 선택
best_tier = max(tier_names, key=lambda t: baseline_results[t]['roc_auc']['cv_mean'])
best_feats = TIERS[best_tier]
print(f'최적 tier (CV AUC 기준): {best_tier}  →  feature 수: {len(best_feats)}')
print(f'  CV AUC: {baseline_results[best_tier]["roc_auc"]["cv_mean"]:.4f} ± {baseline_results[best_tier]["roc_auc"]["cv_std"]:.4f}')

In [ ]:
# GridSearchCV — 과적합 제어 파라미터 집중
X_tune = train[best_feats]
y_tune = train['target']

cat = get_cat_cols(best_feats)
num = [c for c in best_feats if c not in cat]

param_grid = {
    'clf__max_depth':        [3, 4, 5, 6, 8, 10, None],
    'clf__min_samples_leaf': [5, 10, 20, 30, 50],
    'clf__min_samples_split':[10, 20, 40],
    'clf__ccp_alpha':        [0.0, 0.001, 0.002, 0.005, 0.01],
    'clf__class_weight':     [None, 'balanced'],
}

dt_base = DecisionTreeClassifier(random_state=RANDOM_STATE)
pipe_tune = build_pipeline(best_feats, dt_base)

# GridSearch는 조합이 많으면 오래 걸릴 수 있음 — RandomizedSearchCV로 1차 탐색
from sklearn.model_selection import RandomizedSearchCV
rs = RandomizedSearchCV(
    pipe_tune, param_grid,
    n_iter=120,
    scoring='roc_auc',
    cv=CV,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=1,
)
rs.fit(X_tune, y_tune)

print('\n최적 파라미터:')
for k, v in rs.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nBest CV AUC (RandomizedSearch): {rs.best_score_:.4f}')

In [ ]:
# 최적 파라미터로 정밀 GridSearch (핵심 파라미터 좁힘)
best_p = rs.best_params_

def narrow(val, options, n=3):
    """val 주변 n개 값 반환 (options 리스트 내에서)"""
    opts = [o for o in options if o is not None]
    if val is None or val not in opts:
        return options
    idx = opts.index(val)
    lo = max(0, idx - 1)
    hi = min(len(opts), idx + 2)
    result = opts[lo:hi]
    if None in options:
        result = result + [None]
    return result

depth_opts_full = [3, 4, 5, 6, 7, 8, 10, 12, None]
leaf_opts_full  = [5, 8, 10, 15, 20, 30, 50]
split_opts_full = [5, 10, 15, 20, 30, 40]
ccp_opts_full   = [0.0, 0.0005, 0.001, 0.0015, 0.002, 0.003, 0.005, 0.008, 0.01]

fine_grid = {
    'clf__max_depth':        narrow(best_p['clf__max_depth'], depth_opts_full),
    'clf__min_samples_leaf': narrow(best_p['clf__min_samples_leaf'], leaf_opts_full),
    'clf__min_samples_split': narrow(best_p['clf__min_samples_split'], split_opts_full),
    'clf__ccp_alpha':        narrow(best_p['clf__ccp_alpha'], ccp_opts_full),
    'clf__class_weight':     [best_p['clf__class_weight']],
}
print('Fine grid:')
for k, v in fine_grid.items():
    print(f'  {k}: {v}')

gs = GridSearchCV(
    build_pipeline(best_feats, DecisionTreeClassifier(random_state=RANDOM_STATE)),
    fine_grid,
    scoring='roc_auc',
    cv=CV,
    n_jobs=-1,
    refit=True,
    verbose=1,
)
gs.fit(X_tune, y_tune)

print('\n최종 최적 파라미터:')
for k, v in gs.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nFinal best CV AUC: {gs.best_score_:.4f}')

In [ ]:
# 튜닝된 모델로 전체 성능 재평가 (train vs CV gap 확인)
best_params_clean = {k.replace('clf__', ''): v for k, v in gs.best_params_.items()}
tuned_dt = DecisionTreeClassifier(random_state=RANDOM_STATE, **best_params_clean)

tuned_res = evaluate(tuned_dt, best_feats)
print_results(f'{best_tier} (Tuned DT)', tuned_res)

print('\n── 베이스라인 대비 개선 (CV AUC) ──')
base_auc  = baseline_results[best_tier]['roc_auc']['cv_mean']
tuned_auc = tuned_res['roc_auc']['cv_mean']
print(f'  Baseline: {base_auc:.4f}  →  Tuned: {tuned_auc:.4f}  (Δ {tuned_auc - base_auc:+.4f})')

gap = tuned_res['roc_auc']['gap']
print(f'  과적합 Gap (Train-CV AUC): {gap:.4f}', '← 양호' if gap < 0.05 else '← 아직 과적합')

## 7. Feature Importance

In [ ]:
# 최적 파이프라인으로 전체 train 학습
final_pipe = gs.best_estimator_
final_pipe.fit(X_tune, y_tune)

# 파이프라인에서 clf 추출
clf_final = final_pipe.named_steps['clf']

# 특성 이름 복원
if 'prep' in final_pipe.named_steps:
    feature_names_out = final_pipe.named_steps['prep'].get_feature_names_out()
else:
    feature_names_out = np.array(best_feats)

importances = clf_final.feature_importances_
fi_df = pd.DataFrame({'feature': feature_names_out, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).reset_index(drop=True)
print(fi_df.to_string(index=False))

In [ ]:
# Feature Importance 막대그래프
top_n = min(20, len(fi_df))
fi_top = fi_df.head(top_n)

# CORE6/CORE7 feature 강조
colors = ['tomato' if f in CORE6 else 'steelblue' for f in fi_top['feature']]

fig, ax = plt.subplots(figsize=(8, max(4, top_n * 0.35)))
ax.barh(fi_top['feature'][::-1], fi_top['importance'][::-1], color=colors[::-1])
ax.set_xlabel('Importance (Gini)')
ax.set_title(f'Feature Importance — {best_tier} Tuned DT\n(red = CORE6 feature)')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='tomato', label='CORE6'), Patch(color='steelblue', label='Other')],
          loc='lower right')
plt.tight_layout()
plt.savefig('feature_importance_dt.png', bbox_inches='tight')
plt.show()
print('Saved: feature_importance_dt.png')

## 8. Tree 구조 시각화

In [ ]:
# export_text (text 형태, 전체 저장)
tree_text = export_text(clf_final, feature_names=list(feature_names_out))
with open('tree_structure.txt', 'w', encoding='utf-8') as f:
    f.write(tree_text)
print('Saved: tree_structure.txt')
# 처음 50줄만 미리보기
print('\n── 트리 구조 (처음 50줄) ──')
print('\n'.join(tree_text.split('\n')[:50]))

In [ ]:
# plot_tree — depth 3까지만 시각화 (가독성)
max_vis_depth = 3
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(
    clf_final,
    max_depth=max_vis_depth,
    feature_names=list(feature_names_out),
    class_names=['0', '1'],
    filled=True,
    rounded=True,
    ax=ax,
    fontsize=8,
)
ax.set_title(f'Decision Tree (depth ≤ {max_vis_depth} 표시) — {best_tier}')
plt.tight_layout()
plt.savefig('tree_plot.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved: tree_plot.png')

## 9. 전체 Tier 비교 요약 테이블

In [ ]:
# Tuned DT를 나머지 tier에도 같은 파라미터로 적용해 비교
summary_rows = []

for tier_name, feats in TIERS.items():
    # 기본 DT
    r_default = baseline_results[tier_name]
    summary_rows.append({
        'tier': tier_name, 'model': 'Default DT',
        'cv_auc':  r_default['roc_auc']['cv_mean'],
        'std_auc': r_default['roc_auc']['cv_std'],
        'train_auc': r_default['roc_auc']['train_mean'],
        'gap_auc': r_default['roc_auc']['gap'],
        'cv_bacc': r_default['balanced_accuracy']['cv_mean'],
        'cv_f1':   r_default['f1_macro']['cv_mean'],
    })

    # 튜닝된 파라미터 적용
    tuned = DecisionTreeClassifier(random_state=RANDOM_STATE, **best_params_clean)
    r_tuned = evaluate(tuned, feats)
    summary_rows.append({
        'tier': tier_name, 'model': 'Tuned DT',
        'cv_auc':  r_tuned['roc_auc']['cv_mean'],
        'std_auc': r_tuned['roc_auc']['cv_std'],
        'train_auc': r_tuned['roc_auc']['train_mean'],
        'gap_auc': r_tuned['roc_auc']['gap'],
        'cv_bacc': r_tuned['balanced_accuracy']['cv_mean'],
        'cv_f1':   r_tuned['f1_macro']['cv_mean'],
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False, float_format='{:.4f}'.format))

## 10. results_decision_tree.md 저장

In [ ]:
from io import StringIO

tuned_auc_str = f"{tuned_res['roc_auc']['cv_mean']:.4f} ± {tuned_res['roc_auc']['cv_std']:.4f}"
tuned_bacc    = f"{tuned_res['balanced_accuracy']['cv_mean']:.4f}"
tuned_f1      = f"{tuned_res['f1_macro']['cv_mean']:.4f}"
tuned_gap     = f"{tuned_res['roc_auc']['gap']:.4f}"

top5_feats = ', '.join(fi_df.head(5)['feature'].tolist())

md_lines = []
md_lines.append('# Decision Tree 결과 요약 — SSU Problem Solving 2026\n')
md_lines.append(f'생성일: 2026-06-02  |  random_state=42\n')

md_lines.append('\n## (a) Tier별 성능표\n')
md_lines.append('| Tier | Model | CV AUC | ±std | Train AUC | Gap | CV Balanced-Acc | CV F1-macro |')
md_lines.append('|------|-------|--------|------|-----------|-----|-----------------|-------------|')
for _, row in summary_df.iterrows():
    md_lines.append(
        f"| {row['tier']} | {row['model']} "
        f"| {row['cv_auc']:.4f} | {row['std_auc']:.4f} "
        f"| {row['train_auc']:.4f} | {row['gap_auc']:.4f} "
        f"| {row['cv_bacc']:.4f} | {row['cv_f1']:.4f} |"
    )

md_lines.append('\n## (b) 최적 하이퍼파라미터\n')
md_lines.append(f'- 최적 tier: **{best_tier}**')
for k, v in best_params_clean.items():
    md_lines.append(f'- `{k}` = `{v}`')
md_lines.append(f'\n최종 CV AUC: **{tuned_auc_str}**  |  Balanced-Acc: {tuned_bacc}  |  F1-macro: {tuned_f1}')
md_lines.append(f'\nTrain-CV Gap (AUC): {tuned_gap}  ← {"양호 (< 0.05)" if float(tuned_gap) < 0.05 else "과적합 주의"}')

md_lines.append('\n## (c) Feature Importance 순위 (상위 10개)\n')
md_lines.append('| 순위 | Feature | Importance |')
md_lines.append('|------|---------|------------|')
for i, (_, row) in enumerate(fi_df.head(10).iterrows(), 1):
    core_tag = ' ★' if row['feature'] in CORE6 else ''
    md_lines.append(f"| {i} | {row['feature']}{core_tag} | {row['importance']:.4f} |")
md_lines.append('\n★ = CORE6 feature')

md_lines.append('\n## (d) 다음 모델로 넘어가기 전 관찰 메모\n')
md_lines.append('- Decision Tree 단일 모델의 한계: CV AUC가 앙상블 모델(HistGB/RF 기준 ~0.67)보다 낮을 가능성.')
md_lines.append('- `max_depth` 제한 및 `ccp_alpha` 가지치기가 과적합 억제에 효과적임.')
md_lines.append('- CORE6 feature들이 importance 상위에 위치하면 사전 EDA 결과와 일관성 있음.')
md_lines.append('- 다음 단계: RandomForest / ExtraTrees / HistGradientBoosting 등 앙상블로 확장.')
md_lines.append('- `evaluate(model, feature_cols)` 함수는 동일하게 재사용 가능.')

md_content = '\n'.join(md_lines)
with open('results_decision_tree.md', 'w', encoding='utf-8') as f:
    f.write(md_content)

print('Saved: results_decision_tree.md')
print(md_content)

## 11. 테스트 예측 & 제출 파일 생성

In [ ]:
if test is not None:
    X_test = test[best_feats]
    pred_proba = final_pipe.predict_proba(X_test)[:, 1]

    submission = pd.DataFrame({
        'id': test['id'],
        'target': pred_proba,
    })

    # sample_submission 컬럼 구조 확인
    sample_sub_path = 'sample_submission.csv'
    if os.path.exists(sample_sub_path):
        sample_sub = pd.read_csv(sample_sub_path)
        submission = submission[sample_sub.columns]
        print('sample_submission 컬럼 구조에 맞춤:', list(sample_sub.columns))

    submission.to_csv('submission_dt.csv', index=False)
    print(f'Saved: submission_dt.csv  ({len(submission)}행)')
    print(submission.head())
else:
    print('test.csv 없음 → 제출 파일 생성 건너뜀')

In [ ]:
# subfinal.csv: 0/1 이진 예측 (submission_dt.csv의 확률값과 별도)
if test is not None:
    pred_binary = final_pipe.predict(X_test)  # 0 또는 1

    subfinal = pd.DataFrame({
        'id': test['id'],
        'target': pred_binary.astype(int),
    })

    # sample_submission 컬럼 순서 맞춤
    if os.path.exists('sample_submission.csv'):
        sample_sub = pd.read_csv('sample_submission.csv')
        subfinal = subfinal[sample_sub.columns]

    subfinal.to_csv('subfinal.csv', index=False)
    print(f'Saved: subfinal.csv  ({len(subfinal)}행, 0/1 이진 예측)')
    print(subfinal["target"].value_counts().sort_index())
    print(subfinal.head())
else:
    print('test.csv 없음 → subfinal.csv 생성 건너뜀')


## 12. 최종 요약

In [ ]:
print('=' * 60)
print('Decision Tree 분석 완료 요약')
print('=' * 60)
print(f'최적 Tier   : {best_tier}  ({len(best_feats)}개 feature)')
print(f'최적 파라미터: {best_params_clean}')
print(f'Tuned CV AUC: {tuned_res["roc_auc"]["cv_mean"]:.4f} ± {tuned_res["roc_auc"]["cv_std"]:.4f}')
print(f'Train AUC   : {tuned_res["roc_auc"]["train_mean"]:.4f}  Gap: {tuned_res["roc_auc"]["gap"]:.4f}')
print(f'Top 5 Feature: {top5_feats}')
print('\n산출물:')
for f in ['baseline_tier_comparison.png', 'feature_importance_dt.png',
          'tree_plot.png', 'tree_structure.txt', 'results_decision_tree.md',
          'submission_dt.csv']:
    exists = '✓' if os.path.exists(f) else '✗'
    print(f'  {exists} {f}')